# Feature Engineering Final untuk CareerPath AI

Notebook ini melakukan **feature engineering final** pada dataset hasil cleaning untuk menghasilkan dataset kaya fitur yang siap digunakan oleh:

- AI Engineer,
- Recommendation System,
- Dashboard Streamlit,
- Analisis lanjutan.

Output akhir notebook ini adalah `job_featured.csv`.

## 1. Import Library

Library yang digunakan berfokus pada manipulasi data, numerik dasar, regular expression, dan pengecekan file.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

try:
    display
except NameError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

## 2. Load Dataset

Dataset utama yang digunakan adalah `job_clean_final.csv`. Jika tersedia, `skill_analysis_ready.csv` juga dimuat untuk membuat fitur berbasis skill.

In [ ]:
JOB_PATH = Path('job_clean_final.csv')
SKILL_PATH = Path('skill_analysis_ready.csv')

if not JOB_PATH.exists():
    raise FileNotFoundError('File job_clean_final.csv tidak ditemukan. Jalankan notebook cleaning terlebih dahulu.')

jobs = pd.read_csv(JOB_PATH)
skill_ready_available = SKILL_PATH.exists()
skill_ready = pd.read_csv(SKILL_PATH) if skill_ready_available else None

print(f'job_clean_final.csv: {jobs.shape[0]:,} baris x {jobs.shape[1]:,} kolom')
print(f'Kolom job_clean_final: {jobs.columns.tolist()}')
display(jobs.head())

if skill_ready_available:
    print(f'skill_analysis_ready.csv: {skill_ready.shape[0]:,} baris x {skill_ready.shape[1]:,} kolom')
    print(f'Kolom skill_analysis_ready: {skill_ready.columns.tolist()}')
    display(skill_ready.head())
else:
    print('skill_analysis_ready.csv tidak ditemukan. Fitur berbasis skill akan dilewati secara otomatis.')

## 3. Validasi Awal

Validasi awal dilakukan untuk memahami missing values, duplikasi, dan tipe data sebelum feature engineering. Dataset siap untuk feature engineering jika kolom utama seperti `job_id`, `title`, `description`, `location`, dan `listed_time` tersedia.

In [ ]:
def audit_data(data, dataset_name):
    return pd.DataFrame({
        'dataset': dataset_name,
        'kolom': data.columns,
        'tipe_data': data.dtypes.astype(str).values,
        'missing_count': data.isna().sum().values,
        'missing_percent': (data.isna().mean() * 100).round(2).values,
        'unique_values': data.nunique(dropna=False).values
    })

audit_jobs = audit_data(jobs, 'job_clean_final')
display(audit_jobs.sort_values('missing_percent', ascending=False))

print(f'Duplicate rows job_clean_final: {jobs.duplicated().sum():,}')
if 'job_id' in jobs.columns:
    print(f'Duplicate job_id job_clean_final: {jobs.duplicated(subset=["job_id"]).sum():,}')

if skill_ready_available:
    audit_skill = audit_data(skill_ready, 'skill_analysis_ready')
    display(audit_skill.sort_values('missing_percent', ascending=False))
    print(f'Duplicate rows skill_analysis_ready: {skill_ready.duplicated().sum():,}')

### Catatan Kesiapan Data

Dataset `job_clean_final.csv` sudah siap untuk feature engineering karena telah memiliki informasi inti lowongan: `job_id`, `title`, `description`, `location`, `remote_allowed`, `formatted_experience_level`, dan tanggal posting. Jika `skill_analysis_ready.csv` tersedia, fitur berbasis skill dapat ditambahkan untuk memperkuat recommendation system dan matching CV.

## 4. Feature Engineering Dasar

Tahap ini membuat fitur dasar dari teks, remote flag, dan level pengalaman.

### A. Title Clean

Kolom `title_clean` dibuat dari `title` dengan lowercase, penghapusan simbol, dan whitespace normalization.

In [ ]:
def clean_simple_text(text):
    """Membersihkan teks dasar untuk fitur NLP dan matching."""
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s\+\#\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

jobs['title_clean'] = jobs['title'].apply(clean_simple_text)
jobs[['title', 'title_clean']].head()

### B. Description Clean

Kolom `description_clean` dibuat dengan merapikan teks dan menghapus spasi ganda. Fitur ini menjadi input penting untuk NLP dan semantic matching.

In [ ]:
jobs['description_clean'] = jobs['description'].apply(clean_simple_text)
jobs[['description', 'description_clean']].head()

### C. Description Length

`desc_length` berisi jumlah kata pada `description_clean`. Secara teknis, deskripsi yang lebih panjang sering menunjukkan informasi pekerjaan yang lebih detail dan kompleks.

In [ ]:
jobs['desc_length'] = jobs['description_clean'].apply(lambda x: len(x.split()) if x else 0)

jobs[['title_clean', 'desc_length']].head()

**Insight teknis:** `desc_length` dapat digunakan sebagai fitur kualitas informasi lowongan. Lowongan dengan deskripsi sangat pendek mungkin kurang informatif untuk model NLP, sedangkan deskripsi yang panjang biasanya menyediakan sinyal tugas, skill, dan kualifikasi yang lebih kaya.

### D. Is Remote

`is_remote` dibuat dari `remote_allowed`. Nilai `1` berarti remote diperbolehkan, sedangkan `0` berarti non-remote atau tidak diketahui.

In [ ]:
jobs['is_remote'] = pd.to_numeric(jobs.get('remote_allowed', 0), errors='coerce').fillna(0).astype(int)
jobs['is_remote'] = np.where(jobs['is_remote'] == 1, 1, 0)

jobs[['remote_allowed', 'is_remote']].head() if 'remote_allowed' in jobs.columns else jobs[['is_remote']].head()

### E. Experience Level Clean

`experience_level_clean` distandarisasi dari `formatted_experience_level`. Jika level kosong atau `not_specified`, notebook mengekstrak sinyal dari `title_clean`, seperti `junior`, `senior`, `intern`, `associate`, `manager`, dan `lead`. Jika tidak ada sinyal, nilai default adalah `mid`.

In [ ]:
def normalize_experience_level(level, title):
    """Menstandarisasi experience level dari kolom level dan title lowongan."""
    level_text = '' if pd.isna(level) else str(level).lower().strip()
    title_text = '' if pd.isna(title) else str(title).lower().strip()
    combined_text = f'{level_text} {title_text}'

    if 'intern' in combined_text or 'internship' in combined_text:
        return 'intern'
    if 'junior' in combined_text or 'entry' in combined_text:
        return 'junior'
    if 'associate' in combined_text:
        return 'associate'
    if 'senior' in combined_text or 'sr ' in combined_text:
        return 'senior'
    if 'manager' in combined_text:
        return 'manager'
    if 'lead' in combined_text or 'head' in combined_text or 'director' in combined_text or 'executive' in combined_text:
        return 'lead'
    if level_text and level_text not in ['not_specified', 'nan', 'none', '']:
        if 'mid' in level_text:
            return 'mid'
    return 'mid'

experience_source = jobs['formatted_experience_level'] if 'formatted_experience_level' in jobs.columns else pd.Series('', index=jobs.index)
jobs['experience_level_clean'] = [
    normalize_experience_level(level, title)
    for level, title in zip(experience_source, jobs['title_clean'])
]

jobs[['formatted_experience_level', 'title_clean', 'experience_level_clean']].head() if 'formatted_experience_level' in jobs.columns else jobs[['title_clean', 'experience_level_clean']].head()

## 5. Feature Engineering Skill

Jika `skill_analysis_ready.csv` tersedia, fitur skill dibuat pada level `job_id`. Jika file tidak tersedia, proses dilewati tanpa menghentikan notebook.

### A. Total Skill per Job

`total_skills` menghitung jumlah skill unik untuk setiap lowongan.

### B. Dominant Skill

`dominant_skill` mengambil skill pertama yang tersedia pada setiap lowongan sebagai representasi skill utama.

### C. Skill Text

`skill_text` menggabungkan seluruh skill per lowongan menjadi string seperti `python sql tableau excel`. Fitur ini sangat berguna untuk NLP, vectorization, dan matching CV.

In [ ]:
if skill_ready_available:
    required_skill_columns = {'job_id', 'skill_name'}
    missing_skill_columns = required_skill_columns - set(skill_ready.columns)

    if missing_skill_columns:
        print(f'Kolom skill tidak lengkap: {missing_skill_columns}. Fitur skill dilewati.')
        skill_features_available = False
    else:
        skill_ready['skill_name_clean'] = skill_ready['skill_name'].apply(clean_simple_text)
        skill_ready = skill_ready[skill_ready['skill_name_clean'] != ''].copy()

        skill_features = (
            skill_ready.groupby('job_id')
            .agg(
                total_skills=('skill_name_clean', 'nunique'),
                dominant_skill=('skill_name_clean', 'first'),
                skill_text=('skill_name_clean', lambda values: ' '.join(sorted(set(values))))
            )
            .reset_index()
        )

        jobs = jobs.merge(skill_features, on='job_id', how='left')
        jobs['total_skills'] = jobs['total_skills'].fillna(0).astype(int)
        jobs['dominant_skill'] = jobs['dominant_skill'].fillna('not_specified')
        jobs['skill_text'] = jobs['skill_text'].fillna('')
        skill_features_available = True

        print(f'Fitur skill berhasil dibuat untuk {skill_features.shape[0]:,} job_id unik.')
        display(skill_features.head())
else:
    skill_features_available = False
    print('skill_analysis_ready.csv tidak tersedia. Fitur total_skills, dominant_skill, dan skill_text dilewati.')

**Insight teknis:** Fitur skill adalah salah satu komponen terpenting untuk CareerPath AI. `skill_text` dapat langsung digunakan untuk perhitungan kemiripan teks antara CV dan lowongan, sedangkan `total_skills` membantu membaca kompleksitas kebutuhan kompetensi pada setiap job.

## 6. Feature Engineering Time

Fitur waktu dibuat dari `listed_time` jika tersedia. Jika `listed_time` kosong, notebook mencoba menggunakan `original_listed_time`.

In [ ]:
def parse_datetime_safe(series):
    """Mengonversi kolom tanggal menjadi datetime dengan aman."""
    return pd.to_datetime(series, errors='coerce')

if 'listed_time' in jobs.columns:
    posted_datetime = parse_datetime_safe(jobs['listed_time'])
elif 'original_listed_time' in jobs.columns:
    posted_datetime = parse_datetime_safe(jobs['original_listed_time'])
else:
    posted_datetime = pd.Series(pd.NaT, index=jobs.index)

if posted_datetime.isna().all() and 'original_listed_time' in jobs.columns:
    posted_datetime = parse_datetime_safe(jobs['original_listed_time'])

jobs['posted_year'] = posted_datetime.dt.year
jobs['posted_month'] = posted_datetime.dt.month
jobs['posted_dayofweek'] = posted_datetime.dt.dayofweek

jobs[['posted_year', 'posted_month', 'posted_dayofweek']].head()

## 7. Feature Engineering Location

Lokasi distandarisasi menjadi `city_clean`, dan dibuat indikator `is_unknown_location` untuk mendeteksi lokasi yang tidak diketahui.

In [ ]:
def clean_city(location):
    """Mengambil kota utama dan menstandarisasi lokasi."""
    if pd.isna(location) or str(location).strip() == '':
        return 'unknown'
    city = str(location).split(',')[0].strip().lower()
    city = re.sub(r'[^a-z0-9\s]', ' ', city)
    city = re.sub(r'\s+', ' ', city).strip()
    return city if city else 'unknown'

jobs['city_clean'] = jobs['location'].apply(clean_city) if 'location' in jobs.columns else 'unknown'
jobs['is_unknown_location'] = np.where(jobs['city_clean'].isin(['unknown', 'not specified', 'not_specified']), 1, 0)

jobs[['location', 'city_clean', 'is_unknown_location']].head() if 'location' in jobs.columns else jobs[['city_clean', 'is_unknown_location']].head()

## 8. Final Dataset Check

Tahap ini memastikan fitur utama telah terbentuk dan menampilkan preview dataset final.

In [ ]:
expected_features = [
    'title_clean',
    'description_clean',
    'desc_length',
    'is_remote',
    'experience_level_clean',
    'posted_month',
    'city_clean'
]

if skill_features_available:
    expected_features.extend(['total_skills', 'dominant_skill', 'skill_text'])

feature_check = pd.DataFrame({
    'feature': expected_features,
    'tersedia': [feature in jobs.columns for feature in expected_features]
})

display(feature_check)
display(jobs.head())

## 9. Validasi Feature

Validasi fitur dilakukan pada missing values fitur baru, distribusi `experience_level_clean`, statistik `desc_length`, dan statistik `total_skills` jika tersedia.

In [ ]:
new_features = [
    'title_clean',
    'description_clean',
    'desc_length',
    'is_remote',
    'experience_level_clean',
    'posted_year',
    'posted_month',
    'posted_dayofweek',
    'city_clean',
    'is_unknown_location'
]

if skill_features_available:
    new_features.extend(['total_skills', 'dominant_skill', 'skill_text'])

feature_missing = pd.DataFrame({
    'feature': new_features,
    'missing_count': jobs[new_features].isna().sum().values,
    'missing_percent': (jobs[new_features].isna().mean() * 100).round(2).values,
    'tipe_data': jobs[new_features].dtypes.astype(str).values
})

display(feature_missing)

print('Distribusi experience_level_clean:')
display(jobs['experience_level_clean'].value_counts().reset_index().rename(columns={'index': 'experience_level_clean', 'experience_level_clean': 'jumlah'}))

print('Statistik desc_length:')
display(jobs['desc_length'].describe())

if skill_features_available:
    print('Statistik total_skills:')
    display(jobs['total_skills'].describe())

## 10. Simpan Output Final

Dataset final disimpan sebagai `job_featured.csv`.

In [ ]:
OUTPUT_PATH = 'job_featured.csv'
jobs.to_csv(OUTPUT_PATH, index=False)

print(f'Dataset feature engineering berhasil disimpan ke: {OUTPUT_PATH}')
print(f'Shape final: {jobs.shape[0]:,} baris x {jobs.shape[1]:,} kolom')

## 11. Dokumentasi Feature Dictionary

Feature dictionary mendokumentasikan fitur baru agar mudah digunakan oleh AI Engineer, Data Analyst, dan tim dashboard.

In [ ]:
feature_descriptions = {
    'title_clean': 'Title pekerjaan yang telah dibersihkan untuk kebutuhan NLP dan matching.',
    'description_clean': 'Deskripsi pekerjaan yang telah dirapikan untuk NLP dan semantic matching.',
    'desc_length': 'Jumlah kata pada description_clean; indikator detail dan kompleksitas lowongan.',
    'is_remote': 'Indikator pekerjaan remote; 1 remote, 0 non-remote atau unknown.',
    'experience_level_clean': 'Level pengalaman standar hasil normalisasi dari experience level dan title.',
    'total_skills': 'Jumlah skill unik yang terkait dengan job_id.',
    'dominant_skill': 'Skill utama atau skill pertama yang terkait dengan job_id.',
    'skill_text': 'Gabungan seluruh skill per job dalam format teks untuk NLP dan CV matching.',
    'posted_year': 'Tahun posting lowongan.',
    'posted_month': 'Bulan posting lowongan.',
    'posted_dayofweek': 'Hari posting dalam angka; 0 Senin sampai 6 Minggu.',
    'city_clean': 'Nama kota/lokasi utama yang telah distandarisasi.',
    'is_unknown_location': 'Indikator lokasi unknown; 1 unknown, 0 tersedia.'
}

available_feature_descriptions = {feature: desc for feature, desc in feature_descriptions.items() if feature in jobs.columns}

feature_dictionary = pd.DataFrame({
    'feature': list(available_feature_descriptions.keys()),
    'tipe_data': [str(jobs[feature].dtype) for feature in available_feature_descriptions.keys()],
    'deskripsi': list(available_feature_descriptions.values())
})

feature_dictionary

## 12. Kesimpulan Akhir

Feature engineering final telah menghasilkan dataset `job_featured.csv` yang lebih kaya dan siap digunakan untuk tahap AI maupun dashboard.

Fitur paling penting untuk AI Engineer adalah:
- `title_clean` dan `description_clean`, karena menjadi input utama NLP matching.
- `skill_text`, karena merepresentasikan kebutuhan skill tiap lowongan dalam format yang mudah di-vectorize.
- `total_skills`, karena memberi sinyal kompleksitas kebutuhan skill.
- `experience_level_clean`, karena membantu personalisasi rekomendasi berdasarkan level karier.
- `city_clean` dan `is_remote`, karena mendukung filtering preferensi pengguna.

Untuk recommendation system, fitur ini membantu membangun skor kecocokan antara CV dan lowongan melalui kombinasi teks pekerjaan, skill, lokasi, remote preference, dan level pengalaman.

Untuk dashboard Streamlit, fitur ini mendukung visualisasi distribusi level pengalaman, peluang remote, tren waktu posting, persebaran kota, kompleksitas deskripsi, dan kebutuhan skill per pekerjaan.